# UAS Kecerdasan Buatan
## Prediksi Risiko Penyakit Jantung Menggunakan Algoritma Decision Tree dan K-Nearest Neighbors (KNN)

**Nama Kelompok:** (isi nama & NIM anggota, maksimal 2 orang)
**Mata Kuliah:** Kecerdasan Buatan
**Tahun:** 2026

Notebook ini berisi seluruh proses pengerjaan proyek klasifikasi risiko penyakit jantung, mulai dari pemahaman data, eksplorasi data (EDA), praproses data, pemodelan dengan 2 algoritma (Decision Tree dan KNN), hingga evaluasi dan kesimpulan.

Laporan lengkap (naratif) tersedia di file `Laporan_uas.md`.

## 1. Import Library

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (confusion_matrix, classification_report,
                              accuracy_score, precision_score, recall_score, f1_score)

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (8,5)

RANDOM_STATE = 42


## 2. Data Understanding

**Sumber data:** Dataset disimulasikan (simulasi) mengikuti struktur dan karakteristik statistik dari dataset publik terkenal *Heart Disease UCI* (Cleveland Heart Disease dataset, tersedia di Kaggle/UCI Machine Learning Repository). Simulasi dilakukan karena keterbatasan akses jaringan pada lingkungan pengerjaan, namun struktur fitur, tipe data, rentang nilai, dan hubungan antar-fitur terhadap target dibuat merepresentasikan kondisi nyata pada domain penyakit jantung.

**Jumlah data:** 505 baris (termasuk beberapa duplikat & missing value yang sengaja disisipkan untuk latihan data cleaning), 14 kolom.

**Deskripsi fitur:**

| Kolom | Deskripsi | Tipe |
|---|---|---|
| age | Usia pasien (tahun) | Numerik |
| sex | Jenis kelamin (1 = laki-laki, 0 = perempuan) | Kategorik (biner) |
| cp | Tipe nyeri dada (0-3) | Kategorik |
| trestbps | Tekanan darah istirahat (mm Hg) | Numerik |
| chol | Kadar kolesterol serum (mg/dl) | Numerik |
| fbs | Gula darah puasa > 120 mg/dl (1=ya, 0=tidak) | Kategorik (biner) |
| restecg | Hasil elektrokardiografi istirahat (0-2) | Kategorik |
| thalach | Detak jantung maksimum tercapai | Numerik |
| exang | Angina akibat olahraga (1=ya, 0=tidak) | Kategorik (biner) |
| oldpeak | Depresi ST akibat olahraga relatif terhadap istirahat | Numerik |
| slope | Kemiringan segmen ST puncak olahraga (0-2) | Kategorik |
| ca | Jumlah pembuluh darah utama yang terwarnai fluoroskopi (0-3) | Kategorik |
| thal | Kondisi thalassemia (1=normal, 2=cacat tetap, 3=cacat reversibel) | Kategorik |
| **target** | **Label: 1 = berisiko penyakit jantung, 0 = tidak berisiko** | **Target (biner)** |

**Tipe masalah:** Klasifikasi biner (binary classification).

In [ ]:
df = pd.read_csv('data/dataset/heart_disease_dataset.csv')
df.head()


In [ ]:
df.info()


In [ ]:
df.describe()


In [ ]:
print("Jumlah baris duplikat:", df.duplicated().sum())
print("\nJumlah missing value per kolom:")
print(df.isna().sum())
print("\nUkuran data:", df.shape)
print("\nDistribusi target:")
print(df['target'].value_counts())


## 3. Exploratory Data Analysis (EDA)

### 3.1 Distribusi Target (Deteksi Data Tidak Seimbang)

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(x='target', data=df, palette='Set2')
plt.title('Distribusi Kelas Target (0 = Tidak Berisiko, 1 = Berisiko)')
plt.xlabel('Target')
plt.ylabel('Jumlah')
plt.show()

print(df['target'].value_counts(normalize=True).round(3) * 100, "%")


**Insight:** Distribusi kelas cukup seimbang (proporsi mendekati 55:45), sehingga tidak diperlukan teknik penanganan *imbalanced class* khusus seperti SMOTE atau oversampling.

### 3.2 Distribusi Fitur Numerik (Histogram)

In [ ]:
num_cols = ['age','trestbps','chol','thalach','oldpeak']
df[num_cols].hist(bins=20, figsize=(12,8), color='steelblue', edgecolor='black')
plt.suptitle('Distribusi Fitur Numerik')
plt.tight_layout()
plt.show()


### 3.3 Perbandingan Fitur terhadap Target (Boxplot)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15,4))
for ax, col in zip(axes, ['age','chol','thalach']):
    sns.boxplot(x='target', y=col, data=df, ax=ax, palette='Set3')
    ax.set_title(f'{col} vs target')
plt.tight_layout()
plt.show()


### 3.4 Korelasi Antar Fitur (Heatmap)

In [ ]:
plt.figure(figsize=(12,9))
corr = df.corr(numeric_only=True)
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Heatmap Korelasi Antar Fitur')
plt.show()


### 3.5 Pairplot Fitur Utama

In [ ]:
sns.pairplot(df[['age','chol','thalach','oldpeak','target']].dropna(), hue='target', palette='husl', corner=True)
plt.suptitle('Pairplot Fitur Utama terhadap Target', y=1.02)
plt.show()


**Insight EDA:**
- Pasien dengan target=1 (berisiko) cenderung memiliki `oldpeak` yang lebih tinggi dan `thalach` (detak jantung maksimum) yang lebih rendah dibanding kelas 0.
- `exang`, `ca`, dan `cp` menunjukkan korelasi yang cukup terlihat terhadap target pada heatmap.
- Tidak ditemukan korelasi ekstrem (multikolinearitas berat) antar fitur numerik, sehingga seluruh fitur numerik dapat dipertahankan untuk pemodelan.
- Ditemukan sejumlah kecil missing value pada `chol` dan `thalach`, serta data duplikat, yang akan ditangani pada tahap Data Preparation.

## 4. Data Preparation

### 4.1 Pembersihan Data (Duplikat & Missing Value)

In [ ]:
# Hapus duplikat
before = df.shape[0]
df = df.drop_duplicates().reset_index(drop=True)
print(f"Baris sebelum: {before}, sesudah hapus duplikat: {df.shape[0]}")

# Imputasi missing value numerik dengan median
for col in ['chol', 'thalach']:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)

print("\nSisa missing value:", df.isna().sum().sum())


### 4.2 Encoding Data Kategorik

Fitur kategorik pada dataset ini (`sex`, `cp`, `fbs`, `restecg`, `exang`, `slope`, `ca`, `thal`) sudah berbentuk numerik (encoded) sejak sumber datanya (mengikuti standar dataset UCI Heart Disease). Karena kategori tersebut bersifat **nominal dengan jumlah kategori kecil**, tidak diperlukan One-Hot Encoding tambahan — nilai numerik ini sudah merepresentasikan kategori secara diskrit dan dapat langsung digunakan oleh Decision Tree maupun KNN (setelah standardisasi khusus untuk KNN).

### 4.3 Standardisasi Data Numerik

In [ ]:
feature_cols = [c for c in df.columns if c != 'target']
X = df[feature_cols]
y = df['target']

print("Fitur:", feature_cols)
print("Jumlah sampel:", X.shape[0])


### 4.4 Split Data (Train-Test)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print("Ukuran data latih:", X_train.shape)
print("Ukuran data uji:", X_test.shape)

# Standardisasi (penting untuk KNN, tidak wajib untuk Decision Tree tapi tidak merugikan)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


## 5. Modeling

**Algoritma yang dipilih:** Decision Tree dan K-Nearest Neighbors (KNN)

**Alasan pemilihan:**
1. **Decision Tree** dipilih karena mudah diinterpretasikan (dapat divisualisasikan sebagai pohon keputusan) dan mampu menangkap hubungan non-linear antar fitur medis tanpa perlu standardisasi data, sehingga hasilnya dapat dijelaskan secara klinis (misalnya "jika `cp`=0 dan `oldpeak`>1.5 maka berisiko").
2. **KNN** dipilih sebagai pembanding karena bekerja berdasarkan kemiripan antar pasien (jarak antar data), yang relevan secara konseptual dalam domain medis (pasien dengan profil klinis mirip cenderung memiliki diagnosis yang mirip pula), dan dapat menunjukkan performa berbeda dibanding pendekatan berbasis aturan seperti Decision Tree.

### 5.1 Model 1: Decision Tree

In [ ]:
dt_model = DecisionTreeClassifier(max_depth=4, random_state=RANDOM_STATE)
dt_model.fit(X_train, y_train)

y_pred_dt = dt_model.predict(X_test)
print("Decision Tree - Akurasi:", accuracy_score(y_test, y_pred_dt).round(3))


### 5.2 Model 2: K-Nearest Neighbors (KNN)

In [ ]:
knn_model = KNeighborsClassifier(n_neighbors=7)
knn_model.fit(X_train_scaled, y_train)

y_pred_knn = knn_model.predict(X_test_scaled)
print("KNN - Akurasi:", accuracy_score(y_test, y_pred_knn).round(3))


### 5.3 Visualisasi Pohon Keputusan

In [ ]:
plt.figure(figsize=(20,10))
plot_tree(dt_model, feature_names=feature_cols, class_names=['Tidak Berisiko','Berisiko'],
          filled=True, rounded=True, fontsize=9)
plt.title('Visualisasi Decision Tree (max_depth=4)')
plt.show()


### 5.4 Feature Importance (Decision Tree)

In [ ]:
importances = pd.Series(dt_model.feature_importances_, index=feature_cols).sort_values(ascending=False)
plt.figure(figsize=(8,5))
sns.barplot(x=importances.values, y=importances.index, palette='viridis')
plt.title('Feature Importance - Decision Tree')
plt.xlabel('Importance')
plt.show()
print(importances)


## 6. Evaluation

### 6.1 Confusion Matrix

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12,5))

cm_dt = confusion_matrix(y_test, y_pred_dt)
sns.heatmap(cm_dt, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Tidak Berisiko','Berisiko'], yticklabels=['Tidak Berisiko','Berisiko'])
axes[0].set_title('Confusion Matrix - Decision Tree')
axes[0].set_xlabel('Prediksi'); axes[0].set_ylabel('Aktual')

cm_knn = confusion_matrix(y_test, y_pred_knn)
sns.heatmap(cm_knn, annot=True, fmt='d', cmap='Greens', ax=axes[1],
            xticklabels=['Tidak Berisiko','Berisiko'], yticklabels=['Tidak Berisiko','Berisiko'])
axes[1].set_title('Confusion Matrix - KNN')
axes[1].set_xlabel('Prediksi'); axes[1].set_ylabel('Aktual')

plt.tight_layout()
plt.show()


### 6.2 Metrik Evaluasi: Accuracy, Precision, Recall, F1-Score

In [ ]:
def get_metrics(y_true, y_pred, model_name):
    return {
        'Model': model_name,
        'Accuracy': round(accuracy_score(y_true, y_pred), 3),
        'Precision': round(precision_score(y_true, y_pred), 3),
        'Recall': round(recall_score(y_true, y_pred), 3),
        'F1-Score': round(f1_score(y_true, y_pred), 3)
    }

results = pd.DataFrame([
    get_metrics(y_test, y_pred_dt, 'Decision Tree'),
    get_metrics(y_test, y_pred_knn, 'KNN')
])
results


In [ ]:
print("=== Classification Report: Decision Tree ===")
print(classification_report(y_test, y_pred_dt, target_names=['Tidak Berisiko','Berisiko']))

print("=== Classification Report: KNN ===")
print(classification_report(y_test, y_pred_knn, target_names=['Tidak Berisiko','Berisiko']))


### 6.3 Perbandingan Visual Performa Model

In [ ]:
results_melted = results.melt(id_vars='Model', var_name='Metrik', value_name='Nilai')
plt.figure(figsize=(9,5))
sns.barplot(data=results_melted, x='Metrik', y='Nilai', hue='Model', palette='Set1')
plt.title('Perbandingan Performa Decision Tree vs KNN')
plt.ylim(0,1)
plt.show()


**Penjelasan kinerja model berdasarkan metrik:**

Berdasarkan hasil run notebook ini (random_state=42, split 80:20):

| Metrik | Decision Tree | KNN (k=7) |
|---|---|---|
| Accuracy | 0.680 | 0.630 |
| Precision | 0.651 | 0.633 |
| Recall | 0.622 | 0.422 |
| F1-Score | 0.636 | 0.507 |

**Model terbaik: Decision Tree.** Decision Tree unggul di seluruh metrik dibandingkan KNN pada dataset ini. Yang paling signifikan adalah selisih **recall** (0.622 vs 0.422) — artinya KNN gagal mendeteksi proporsi pasien berisiko yang jauh lebih besar (false negative lebih banyak: 26 vs 17 pada confusion matrix). Dalam konteks medis, recall yang rendah pada kelas "Berisiko" berbahaya karena berarti banyak pasien berisiko yang justru diprediksi aman.

Kemungkinan penyebab KNN berkinerja lebih rendah:
- KNN sensitif terhadap fitur yang kurang informatif/noise (curse of dimensionality) karena seluruh 13 fitur diberi bobot jarak yang sama, sedangkan Decision Tree secara otomatis memilih fitur paling informatif (terlihat dari feature importance: `age`, `ca`, `trestbps`, dan `oldpeak` mendominasi, sementara `chol`, `fbs`, `slope`, `thal` bahkan tidak digunakan sama sekali oleh pohon).
- Pemilihan k=7 mungkin belum optimal; nilai k lain berpotensi memberi hasil berbeda (perlu tuning lebih lanjut).

Decision Tree juga memiliki keunggulan interpretability: pohon keputusan dapat dibaca langsung sebagai aturan klinis (lihat visualisasi pohon dan feature importance di atas).

## 7. Kesimpulan dan Rekomendasi

1. **Ringkasan hasil modeling dan evaluasi:** Kedua model (Decision Tree dan KNN) berhasil dilatih untuk mengklasifikasikan risiko penyakit jantung berdasarkan 13 fitur klinis. Decision Tree mencapai accuracy 0.680 dan F1-score 0.636, sedangkan KNN (k=7) mencapai accuracy 0.630 dan F1-score 0.507.
2. **Apakah tujuan proyek tercapai?** Ya. Tujuan untuk membangun dan membandingkan model prediksi risiko penyakit jantung menggunakan dua algoritma berbeda telah tercapai, dengan **Decision Tree sebagai model terbaik** pada dataset ini.
3. **Kelebihan dan keterbatasan model:**
   - *Decision Tree*: mudah diinterpretasikan (dapat divisualisasikan sebagai aturan), secara otomatis menyeleksi fitur penting (age, ca, trestbps, oldpeak), namun berisiko overfitting bila kedalaman pohon tidak dibatasi.
   - *KNN*: sederhana secara konsep, namun performanya lebih rendah di sini karena sensitif terhadap fitur kurang informatif dan pemilihan nilai k yang belum dioptimalkan.
   - Dataset yang digunakan bersifat simulasi berskala menengah (500 baris setelah pembersihan), sehingga generalisasi ke populasi pasien nyata masih terbatas.
4. **Rekomendasi perbaikan:**
   - Menggunakan dataset asli (misalnya UCI Heart Disease Dataset dari Kaggle) dengan jumlah data lebih besar dan tervalidasi secara medis.
   - Melakukan hyperparameter tuning (GridSearchCV) untuk kedua model, termasuk pencarian nilai k optimal untuk KNN dan kedalaman optimal untuk Decision Tree.
   - Mencoba algoritma tambahan seperti Random Forest, SVM, atau Gradient Boosting untuk perbandingan lebih luas.
   - Menerapkan cross-validation agar evaluasi lebih robust terhadap variasi data.